# 🔍 Exploratory Data Analysis — Ember Yearly Electricity
**Dataset:** `europe_yearly_full_release_long_format.csv`  
**Course:** Exploratory Data Analysis  
**Description:** Country-level annual electricity mix and emissions intensity from Ember Climate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.facecolor'] = '#1E293B'
plt.rcParams['figure.facecolor'] = '#0F172A'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.titlecolor'] = '#38BDF8'
plt.rcParams['grid.color'] = '#334155'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.4

PALETTE = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED',
           '#0891B2','#DB2777','#65A30D','#EA580C','#6366F1']
ACCENT = '#38BDF8'

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/europe_yearly_full_release_long_format.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Columns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
print('\nYear range:', df['Year'].min(), '—', df['Year'].max())
print('Countries:', df['Area'].nunique())
print('Variables:', df['Variable'].nunique())

## 2. Data Cleaning & Preprocessing

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

# Convert dtypes
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
df['YoY absolute change'] = pd.to_numeric(df['YoY absolute change'], errors='coerce')
df['YoY % change'] = pd.to_numeric(df['YoY % change'], errors='coerce')

# Flag membership
for col in ['EU','OECD','G20','G7','ASEAN']:
    df[col] = df[col].fillna(0).astype(int)

# Country-only flag
df['is_country'] = df['Area type'].str.lower().str.contains('country', na=False)
countries_df = df[df['is_country']].copy()

print('\nCleaning complete. Shape:', df.shape)
print('Country rows:', countries_df.shape[0])

## 3. Univariate Analysis

In [ ]:
# Distribution of CO2 intensity
co2_data = countries_df[countries_df['Variable']=='CO2 intensity']['Value'].dropna()
fig, ax = plt.subplots()
ax.hist(co2_data, bins=40, color=ACCENT, edgecolor='#0F172A', alpha=0.85)
ax.axvline(co2_data.mean(), color='#F59E0B', linestyle='--', lw=2, label=f'Mean: {co2_data.mean():.0f}')
ax.axvline(co2_data.median(), color='#34D399', linestyle=':', lw=2, label=f'Median: {co2_data.median():.0f}')
ax.set_xlabel('CO₂ Intensity (gCO₂e/kWh)')
ax.set_ylabel('Frequency')
ax.set_title('Histogram: CO₂ Intensity Distribution', fontweight='bold')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()
print(co2_data.describe())

## 4. Bivariate Analysis

In [ ]:
# Renewables vs CO2 intensity scatter (latest available year)
latest = countries_df['Year'].max()
ren = countries_df[(countries_df['Variable']=='Renewables') & (countries_df['Unit']=='%') & (countries_df['Year']==latest)][['Area','Value']].rename(columns={'Value':'Ren'})
co2 = countries_df[(countries_df['Variable']=='CO2 intensity') & (countries_df['Year']==latest)][['Area','Value']].rename(columns={'Value':'CO2'})
merged = ren.merge(co2, on='Area').dropna()

fig, ax = plt.subplots()
sc = ax.scatter(merged['Ren'], merged['CO2'], c=merged['CO2'], cmap='RdYlGn_r', s=100, edgecolors='#0F172A', lw=0.8, alpha=0.9)
for _, row in merged.iterrows():
    ax.annotate(row['Area'], (row['Ren'], row['CO2']), fontsize=7, alpha=0.8, xytext=(3,3), textcoords='offset points')
plt.colorbar(sc, ax=ax, label='CO₂ Intensity')
z = np.polyfit(merged['Ren'], merged['CO2'], 1)
p = np.poly1d(z)
xs = np.linspace(merged['Ren'].min(), merged['Ren'].max(), 100)
ax.plot(xs, p(xs), '--', color='#F59E0B', lw=1.5, label='Trend')
ax.set_xlabel('Renewables (%)')
ax.set_ylabel('CO₂ Intensity (gCO₂e/kWh)')
ax.set_title(f'Renewables vs CO₂ Intensity — {latest}', fontweight='bold')
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

## 5. Multivariate Analysis — Correlation Heatmap

In [ ]:
key_vars = ['Demand','Renewables','Fossil','Nuclear','CO2 intensity','Solar','Wind','Hydro']
year = 2022
sub = countries_df[(countries_df['Year']==year) & (countries_df['Variable'].isin(key_vars))]
pivot = sub.pivot_table(index='Area', columns='Variable', values='Value', aggfunc='mean').dropna(thresh=3)
corr = pivot.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='#0F172A', ax=ax,
            annot_kws={'size':9})
ax.set_title(f'Correlation Matrix of Electricity Variables ({year})', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Time Series — Renewables Growth

In [ ]:
focus = ['Germany','France','Poland','Norway','Spain','United Kingdom']
sub = countries_df[(countries_df['Variable']=='Renewables') & (countries_df['Unit']=='%') & (countries_df['Area'].isin(focus))]
fig, ax = plt.subplots()
for i, c in enumerate(focus):
    data = sub[sub['Area']==c].sort_values('Year')
    ax.plot(data['Year'], data['Value'], marker='o', markersize=3, lw=2,
            color=PALETTE[i], label=c)
ax.set_xlabel('Year')
ax.set_ylabel('Renewables (%)')
ax.set_title('Renewables Share Over Time — Selected Countries', fontweight='bold')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()

## 7. Key Insights Summary

In [ ]:
print('=== KEY FINDINGS ===')
latest = countries_df['Year'].max()
co2_latest = countries_df[(countries_df['Variable']=='CO2 intensity') & (countries_df['Year']==latest)]
print(f'Cleanest grid ({latest}):', co2_latest.nsmallest(3,'Value')[['Area','Value']].to_string(index=False))
print(f'Dirtiest grid ({latest}):', co2_latest.nlargest(3,'Value')[['Area','Value']].to_string(index=False))

ren_latest = countries_df[(countries_df['Variable']=='Renewables') & (countries_df['Unit']=='%') & (countries_df['Year']==latest)]
print(f'\nTop renewables ({latest}):', ren_latest.nlargest(3,'Value')[['Area','Value']].to_string(index=False))

# CO2 trend for EU aggregate
eu_co2 = df[(df['Area']=='EU') & (df['Variable']=='CO2 intensity')].sort_values('Year')
if not eu_co2.empty:
    first_val = eu_co2.iloc[0]['Value']
    last_val  = eu_co2.iloc[-1]['Value']
    pct_change = ((last_val - first_val) / first_val) * 100
    print(f'\nEU CO₂ intensity change {eu_co2.iloc[0]["Year"]}→{eu_co2.iloc[-1]["Year"]}: {pct_change:.1f}%')